# M9d · Celeb-DF v2 — face-crop preprocessing + cross-dataset test (no training)

**Goal.** Two tasks in one notebook:

**Part A (sections 1–6):** Turn the pre-extracted Celeb-DF v2 frames into face crops made exactly like the FF++ training crops (M0) and the DFD/UADFV crops (M8a/M9a). No training.

**Part B (sections 7–12):** Cross-dataset evaluation — same 9 checkpoints as M8b/M9b (frozen CLIP ViT-L/14 + LayerNorms + head, **104,450** trainable at test time). Video score = mean frame probability; metric = video AUC. Paper reference: **GenD (CLIP ViT-L/14) Celeb-DF-v2 ≈ 89.4** (Yermakov et al., WACV 2026, Table 4, mean of 5 seeds).

**Key difference from M8a/M9a.** Celeb-DF v2 ships with frames already extracted into folder trees:
- Real  : `original_sequences/youtube/c23/frames/<video_id>/<frame_files>`
- Fake  : `manipulated_sequences/CelebDF_Synthesis/c23/frames/<video_id>/<frame_files>`

No video decoding is needed. We sample 32 evenly-spaced frame files per folder, apply the JPEG round-trip measured in M8a, then run MTCNN.

**Dataset size:** 888 real video folders + 5,645 fake video folders.

| Section | What | Run time |
|---|---|---|
| 1 | settings, imports, environment | seconds |
| 2 | inventory: list folders, label, frame counts | ~1 min |
| 3 | JPEG settings (reused from M8a) + quality sanity check | seconds |
| 4 | face extraction: helpers, smoke test, full run (resumable) | ~15–45 min |
| 5 | extraction status, failures, faces per frame | seconds |
| 6 | build index CSV, summary, example crops | seconds |
| 7 | model loading, parameter report, FF++ load checks | ~5 min |
| 8 | load Celeb-DF crops into RAM (~31 GB) | ~5–15 min |
| 9 | predict every model (resumable) | ~5 min per model |
| 10 | main table vs paper, bootstrap intervals, paired differences | ~1 min |
| 11 | cross-dataset λ curve (FF++ unseen / DFD / UADFV / Celeb-DF v2) | seconds |
| 12 | ROC curves, final summary JSON | seconds |

# Part A · Face-crop preprocessing

## 1. Setup

### 1.1 Paths and settings

Crop settings are copied from M0/M8a and must not be changed, otherwise Celeb-DF crops would differ from the training crops.

In [ ]:
# 1.1 Paths and settings
from pathlib import Path

# ── inputs (read only) ──────────────────────────────────────────────────────
CELEB_ROOT       = Path("/nlsasfs/home/isea/isea64/celebdf_v2/Celeb-DF_aligned/Celeb-DF_aligned")
REAL_FRAMES_ROOT = CELEB_ROOT / "original_sequences" / "youtube" / "c23" / "frames"
FAKE_FRAMES_ROOT = CELEB_ROOT / "manipulated_sequences" / "CelebDF_Synthesis" / "c23" / "frames"
M8A_PIPELINE     = Path("/nlsasfs/home/isea/isea64/GBLA-Deepfake Detection/faces_dfd/pipeline.json")

# ── outputs ─────────────────────────────────────────────────────────────────
DATA_ROOT  = Path("/nlsasfs/home/isea/isea64/GBLA-Deepfake Detection")
FACES_ROOT = DATA_ROOT / "faces_celebdf"
INDEX_CSV  = DATA_ROOT / "index" / "celebdf_v2.csv"
RUN_DIR    = DATA_ROOT / "runs" / "m9d"

EXPECTED_REAL, EXPECTED_FAKE = 888, 5645

# ── crop settings (identical to M0 / M8a / M9a) ─────────────────────────────
FRAMES_PER_VIDEO = 32
CROP_SCALE       = 1.3
IMAGE_SIZE       = 224
IMAGE_FORMAT     = "png"
MIN_FACE_PROB    = 0.9
DETECTOR_BATCH   = 16
LOAD_WORKERS     = 8
RETRY_FAILED     = False
SEED             = 0
FRAME_EXTS       = {".jpg", ".jpeg", ".png"}

assert REAL_FRAMES_ROOT.is_dir(), f"real frames root not found: {REAL_FRAMES_ROOT}"
assert FAKE_FRAMES_ROOT.is_dir(), f"fake frames root not found: {FAKE_FRAMES_ROOT}"
assert M8A_PIPELINE.exists(),     f"run M8a first: {M8A_PIPELINE}"

for d in (FACES_ROOT, INDEX_CSV.parent, RUN_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("real frames root :", REAL_FRAMES_ROOT)
print("fake frames root :", FAKE_FRAMES_ROOT)
print("outputs          :", FACES_ROOT, "|", INDEX_CSV, "|", RUN_DIR)

### 1.2 Imports and environment

In [ ]:
# 1.2 Imports and environment
import io, json, os, platform, random, time
from collections import Counter, deque
from concurrent.futures import ThreadPoolExecutor
from importlib.metadata import version
from itertools import islice

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from PIL import Image
from tqdm.auto import tqdm
from facenet_pytorch import MTCNN

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

env = {
    "python":          platform.python_version(),
    "torch":           torch.__version__,
    "facenet-pytorch": version("facenet-pytorch"),
    "pillow":          Image.__version__,
    "GPU":             torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
}
display(pd.Series(env, name="value").to_frame())
(RUN_DIR / "environment.json").write_text(json.dumps(env, indent=2))

## 2. Inventory

Scan the two frame-folder trees, label each video folder, record how many frame files it contains and the first/last frame filenames (for debugging).
Expected: 888 real + 5,645 fake, no duplicate video IDs, no empty folders.

In [ ]:
# 2.1 Find and label the video folders
rows = []
for label, frames_root in ((0, REAL_FRAMES_ROOT), (1, FAKE_FRAMES_ROOT)):
    for vid_dir in sorted(p for p in frames_root.iterdir() if p.is_dir()):
        frame_files = sorted(
            f for f in vid_dir.iterdir()
            if f.is_file() and f.suffix.lower() in FRAME_EXTS
        )
        rows.append({
            "video_id":    vid_dir.name,
            "label":       label,
            "frames_dir":  str(vid_dir),
            "n_frames":    len(frame_files),
            "first_frame": frame_files[0].name  if frame_files else "",
            "last_frame":  frame_files[-1].name if frame_files else "",
        })

videos = pd.DataFrame(rows)
n_real = int((videos.label == 0).sum())
n_fake = int((videos.label == 1).sum())
dups   = int(videos.video_id.duplicated().sum())
empty  = int((videos.n_frames == 0).sum())

print(f"real: {n_real} (expected {EXPECTED_REAL})   "
      f"fake: {n_fake} (expected {EXPECTED_FAKE})")
print(f"duplicate video IDs: {dups}   empty folders: {empty}")
print("\nframe count per class:")
display(videos.groupby("label").n_frames.describe().round(1))

videos.to_csv(RUN_DIR / "videos.csv", index=False)

assert dups == 0, "duplicate video IDs — stop and investigate"
assert (n_real, n_fake) == (EXPECTED_REAL, EXPECTED_FAKE), (
    f"counts differ from expected ({EXPECTED_REAL} real, {EXPECTED_FAKE} fake); stop and report"
)
if empty:
    print("WARNING: empty video folders:")
    display(videos[videos.n_frames == 0][["video_id", "label", "frames_dir"]])

## 3. JPEG settings

Our FF++ training crops were cut from JPEG frames in `frames_all`, not raw pixels. M8a measured the JPEG quality and chroma subsampling of those frames and confirmed a round-trip at those settings brings decoded video frames closer to the originals.

Celeb-DF v2 frames are also JPEG, so we reuse M8a's settings for domain consistency. We also estimate the actual JPEG quality of a Celeb-DF sample and print a warning if it differs by > 5 quality points (we still use M8a's value).

In [ ]:
# 3. JPEG settings from M8a + sanity check
m8a          = json.loads(M8A_PIPELINE.read_text())
JPEG_QUALITY = m8a["jpeg_quality"]
SUBSAMPLING  = m8a["subsampling"]
print(f"from M8a: JPEG quality {JPEG_QUALITY}, "
      f"chroma subsampling code {SUBSAMPLING} (0=4:4:4, 1=4:2:2, 2=4:2:0)")

# Estimate quality from quantization tables in a small sample of Celeb-DF frames
STD_LUMA = [16, 11, 10, 16, 24, 40, 51, 61, 12, 12, 14, 19, 26, 58, 60, 55,
            14, 13, 16, 24, 40, 57, 69, 56, 14, 17, 22, 29, 51, 87, 80, 62,
            18, 22, 37, 56, 68, 109, 103, 77, 24, 35, 55, 64, 81, 104, 113, 92,
            49, 64, 78, 87, 103, 121, 120, 101, 72, 92, 95, 98, 112, 100, 103, 99]

def jpeg_quality_est(path):
    with Image.open(path) as im:
        tables = getattr(im, "quantization", None)
    if not tables:
        return None
    scale = 100 * sum(tables[0]) / sum(STD_LUMA)
    q = (200 - scale) / 2 if scale <= 100 else 5000 / scale
    return int(round(min(max(q, 1), 100)))

rng_q       = random.Random(SEED)
sample_dirs = rng_q.sample(videos.frames_dir.tolist(), min(30, len(videos)))
q_vals = []
for d in sample_dirs:
    jpgs = [f for f in Path(d).iterdir() if f.suffix.lower() in (".jpg", ".jpeg")]
    for f in rng_q.sample(jpgs, min(5, len(jpgs))):
        q = jpeg_quality_est(f)
        if q is not None:
            q_vals.append(q)

if q_vals:
    celeb_q = int(round(sum(q_vals) / len(q_vals)))
    print(f"Celeb-DF frame quality estimate: {celeb_q} (from {len(q_vals)} sampled frames)")
    if abs(celeb_q - JPEG_QUALITY) > 5:
        print(f"WARNING: Celeb-DF quality ({celeb_q}) differs from M8a ({JPEG_QUALITY}) "
              f"by > 5 — still using M8a value for domain consistency")
    else:
        print("OK: within 5 quality points of M8a value")
else:
    print("No JPEG frames sampled (all PNG?) — using M8a quality as-is")

USE_JPEG = True
print(f"JPEG round-trip: ON (quality={JPEG_QUALITY}, subsampling={SUBSAMPLING})")

## 4. Face extraction

**Per video:** sample 32 evenly-spaced frame files → load as RGB → JPEG round-trip → MTCNN on every frame → largest confident face; missing frames take the nearest detected box → 1.3× square → 224×224 PNG at `faces_celebdf/<real|fake>/<video_id>/<key>.png`. Key = position 0–31; original filename stored in box file.

**Resume safety (as M8a/M9a).** `_done` / `_failed` markers per video, boxes saved before crops, temp-file-then-rename writes, errors logged to `faces_celebdf/extraction_log.tsv`.

### 4.1 Helpers

In [ ]:
# 4.1 Helpers
PIPELINE = {
    "frames_per_video": FRAMES_PER_VIDEO,
    "crop_scale":       CROP_SCALE,
    "image_size":       IMAGE_SIZE,
    "min_face_prob":    MIN_FACE_PROB,
    "jpeg_quality":     JPEG_QUALITY if USE_JPEG else None,
    "subsampling":      SUBSAMPLING  if USE_JPEG else None,
    "frame_source":     "pre-extracted",
    "fake_boxes":       "independent",
}
PIPELINE_FILE = FACES_ROOT / "pipeline.json"
if PIPELINE_FILE.exists():
    saved = json.loads(PIPELINE_FILE.read_text())
    assert saved == PIPELINE, (
        f"faces_celebdf was made with different settings:\n{saved}\nvs now:\n{PIPELINE}"
    )
else:
    PIPELINE_FILE.write_text(json.dumps(PIPELINE, indent=2))
print("pipeline:", PIPELINE)

BOX_DIR     = FACES_ROOT / "_boxes"
BOX_DIR.mkdir(parents=True, exist_ok=True)
SAVE_FORMAT = Image.registered_extensions()["." + IMAGE_FORMAT]
VIDEO       = videos.set_index("video_id")


def out_dir_of(vid):
    return FACES_ROOT / ("fake" if VIDEO.at[vid, "label"] else "real") / vid


def sample_frame_files(frames_dir, k=FRAMES_PER_VIDEO):
    """Select k evenly-spaced frame files from a directory, sorted by name."""
    files = sorted(
        f for f in Path(frames_dir).iterdir()
        if f.is_file() and f.suffix.lower() in FRAME_EXTS
    )
    if not files:
        return []
    n       = len(files)
    indices = sorted(set(
        np.linspace(0, n - 1, min(k, n)).round().astype(int).tolist()
    ))
    return [(i, files[i]) for i in indices]


def jpeg_roundtrip(rgb):
    if not USE_JPEG or JPEG_QUALITY is None:
        return rgb
    buf = io.BytesIO()
    kw  = {"quality": JPEG_QUALITY}
    if SUBSAMPLING is not None:
        kw["subsampling"] = SUBSAMPLING
    Image.fromarray(rgb).save(buf, format="JPEG", **kw)
    return np.array(Image.open(io.BytesIO(buf.getvalue())).convert("RGB"))


def pick_face(boxes, probs):
    if boxes is None:
        return None, 0
    confident = [b for b, p in zip(boxes, probs)
                 if p is not None and p >= MIN_FACE_PROB]
    if not confident:
        return None, 0
    best = max(confident, key=lambda b: (b[2] - b[0]) * (b[3] - b[1]))
    return [float(v) for v in best], len(confident)


def fill_missing(raw):
    found = [i for i, b in raw.items() if b is not None]
    if not found:
        return None
    return {i: b if b is not None else raw[min(found, key=lambda j: abs(j - i))]
            for i, b in raw.items()}


def square_box(box, width, height):
    x0, y0, x1, y1 = box
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    side   = min(max(x1 - x0, y1 - y0) * CROP_SCALE, width, height)
    left   = int(round(min(max(cx - side / 2, 0), width  - side)))
    top    = int(round(min(max(cy - side / 2, 0), height - side)))
    s      = int(round(side))
    return [left, top, left + s, top + s]


def crop_path(out_dir, key):
    return out_dir / f"{key:04d}.{IMAGE_FORMAT}"


def save_crop(frame, box, out_path):
    x0, y0, x1, y1 = box
    tmp = out_path.with_name(out_path.name + ".tmp")
    Image.fromarray(frame[y0:y1, x0:x1]).resize(
        (IMAGE_SIZE, IMAGE_SIZE), Image.BICUBIC
    ).save(tmp, format=SAVE_FORMAT)
    os.replace(tmp, out_path)


def write_json(path, obj):
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj))
    os.replace(tmp, path)


def load_boxes(vid):
    p = BOX_DIR / f"{vid}.json"
    return json.loads(p.read_text()) if p.exists() else None


def missing_crops(out_dir, keys):
    return [k for k in sorted(keys) if not crop_path(out_dir, k).exists()]


def video_state(out_dir):
    if (out_dir / "_done").exists():
        return "done"
    if (out_dir / "_failed").exists() and not RETRY_FAILED:
        return "failed"
    return "todo"


def mark_done(out_dir, n_missing=0):
    (out_dir / "_failed").unlink(missing_ok=True)
    (out_dir / "_done").write_text(str(n_missing))


def mark_failed(out_dir, reason):
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "_failed").write_text(reason)


def log_event(video_id, message):
    with open(FACES_ROOT / "extraction_log.tsv", "a") as f:
        f.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')}\t{video_id}\t{message}\n")


def bounded_map(fn, items, workers):
    items = iter(items)
    with ThreadPoolExecutor(workers) as pool:
        pending = deque(pool.submit(fn, x) for x in islice(items, 2 * workers))
        while pending:
            result = pending.popleft().result()
            for x in islice(items, 1):
                pending.append(pool.submit(fn, x))
            yield result

### 4.2 Detection and extraction loop

In [ ]:
# 4.2 Detection and extraction loop
mtcnn = MTCNN(keep_all=True, device=DEVICE, post_process=False)


def load_video_frames(vid):
    """Background thread: read the frames this video still needs.
    Returns (vid, status, payload_frames, boxes_meta).
    """
    out_dir = out_dir_of(vid)
    state   = video_state(out_dir)
    if state != "todo":
        return vid, state, None, None
    try:
        boxes = load_boxes(vid)
        if boxes is not None:
            # resume: only load crops still missing
            todo_keys = missing_crops(out_dir, [int(k) for k in boxes["boxes"]])
            frames = {}
            for k in todo_keys:
                fname = boxes["src_file"].get(str(k))
                if fname:
                    p = Path(VIDEO.at[vid, "frames_dir"]) / fname
                    if p.exists():
                        try:
                            frames[k] = jpeg_roundtrip(
                                np.array(Image.open(p).convert("RGB"))
                            )
                        except Exception as e:
                            log_event(vid, f"resume frame read error key={k}: {e!r}")
            return vid, "resume", frames, boxes

        # new: sample frame files
        sampled = sample_frame_files(VIDEO.at[vid, "frames_dir"])
        if not sampled:
            return vid, "empty", "no frame files found", None
        frames, src_file = {}, {}
        for key, fpath in sampled:
            try:
                frames[key]   = jpeg_roundtrip(np.array(Image.open(fpath).convert("RGB")))
                src_file[key] = fpath.name
            except Exception as e:
                log_event(vid, f"frame read error key={key}: {e!r}")
        meta = {"src_file": {str(k): v for k, v in src_file.items()}}
        return vid, "new", frames, meta

    except Exception as e:
        return vid, "error", repr(e), None


def detect_and_save_boxes(vid, frames, meta):
    """Run MTCNN, fill gaps, write box JSON, return updated meta or None."""
    keys = sorted(frames)
    raw, n_faces = {}, {}
    for start in range(0, len(keys), DETECTOR_BATCH):
        chunk           = keys[start:start + DETECTOR_BATCH]
        boxes_det, probs_det = mtcnn.detect(
            [Image.fromarray(frames[k]) for k in chunk]
        )
        for k, b, p in zip(chunk, boxes_det, probs_det):
            raw[k], n_faces[k] = pick_face(b, p)
    filled = fill_missing(raw)
    if filled is None:
        return None
    h, w = frames[keys[0]].shape[:2]
    sq   = {k: square_box(filled[k], w, h) for k in keys}
    meta.update({
        "size":    [w, h],
        "boxes":   {str(k): b for k, b in sq.items()},
        "n_faces": {str(k): n for k, n in n_faces.items()},
    })
    write_json(BOX_DIR / f"{vid}.json", meta)
    return meta


def run_videos(vids):
    stats = Counter()
    for vid, status, payload, meta in tqdm(
        bounded_map(load_video_frames, vids, LOAD_WORKERS),
        total=len(vids), desc="videos"
    ):
        out_dir = out_dir_of(vid)
        if status in ("done", "failed"):
            stats[f"skipped ({status} earlier)"] += 1
            continue
        if status in ("error", "empty"):
            stats[f"failed: {status}"] += 1
            mark_failed(out_dir, status)
            log_event(vid, f"failed: {status}: {payload}")
            continue
        try:
            frames = payload
            if status == "new":
                if not frames:
                    mark_failed(out_dir, "unreadable")
                    log_event(vid, "failed: unreadable")
                    stats["failed: unreadable"] += 1
                    continue
                meta = detect_and_save_boxes(vid, frames, meta)
                if meta is None:
                    mark_failed(out_dir, "no_face")
                    log_event(vid, "failed: no_face")
                    stats["failed: no_face"] += 1
                    continue

            out_dir.mkdir(parents=True, exist_ok=True)
            for k, frame in frames.items():
                save_crop(frame, meta["boxes"][str(k)], crop_path(out_dir, k))
            keys = [int(k) for k in meta["boxes"]]
            left = missing_crops(out_dir, keys)
            if len(left) == len(keys):
                mark_failed(out_dir, "unreadable")
                log_event(vid, "failed: unreadable after crop")
                stats["failed: unreadable"] += 1
                continue
            mark_done(out_dir, len(left))
            stats["extracted" if status == "new" else "resumed"] += 1
        except Exception as e:
            stats["error (retried next run)"] += 1
            log_event(vid, f"error: {e!r}")
    print(dict(stats))
    return stats

### 4.3 Smoke test: 1 real + 1 fake video folder

Look at the crops: one face per crop, centred, similar framing to FF++ crops. Uses the same output folders, so the full run skips these videos.

In [ ]:
# 4.3 Smoke test
rng_pick = random.Random(SEED)
pick_real = rng_pick.choice(videos[videos.label == 0].video_id.tolist())
pick_fake = rng_pick.choice(videos[videos.label == 1].video_id.tolist())
pick = [pick_real, pick_fake]

t0 = time.time()
run_videos(pick)
per_v = (time.time() - t0) / max(len(pick), 1)
print(f"{per_v:.1f} s per video -> full run ~{per_v * len(videos) / 60:.0f} min "
      f"(upper bound; {LOAD_WORKERS} threads run in parallel)")

fig, axes = plt.subplots(len(pick), 8, figsize=(16, 2.4 * len(pick)))
for row_axes, vid in zip(axes, pick):
    crops = sorted(out_dir_of(vid).glob(f"*.{IMAGE_FORMAT}"))
    meta  = load_boxes(vid) or {"n_faces": {}}
    for ax, c in zip(row_axes, crops[::4]):
        ax.imshow(Image.open(c)); ax.axis("off")
        ax.set_title(
            f"{c.stem} faces={meta['n_faces'].get(str(int(c.stem)), '?')}",
            fontsize=7
        )
    label_str = "FAKE" if VIDEO.at[vid, "label"] else "REAL"
    row_axes[0].text(
        -20, IMAGE_SIZE / 2, f"{label_str} {vid[:30]}",
        rotation=90, va="center", ha="right", fontsize=7
    )
plt.tight_layout(); plt.show()

### 4.4 Full run

Resumable: if the session ends, rerun sections 1–3, 4.1, 4.2 and this cell. Finished videos are skipped. Real videos run first.

In [ ]:
# 4.4 Full run
t0 = time.time()
order = videos.sort_values(["label", "video_id"]).video_id.tolist()
run_stats = run_videos(order)
print(f"took {(time.time() - t0) / 60:.1f} min")

## 5. Extraction status

Expected: all 6,533 videos `done`, very few failures (Celeb-DF is a clean dataset). `frames with ≥2 faces` should be low (most Celeb-DF videos show one person at a time).

In [ ]:
# 5. Status
status_rows = []
for r in videos.itertuples():
    d     = out_dir_of(r.video_id)
    state = ("done"   if (d / "_done").exists()   else
             "failed" if (d / "_failed").exists() else "todo")
    meta  = load_boxes(r.video_id) or {}
    nf    = list(meta.get("n_faces", {}).values())
    status_rows.append({
        "video_id":          r.video_id,
        "label":             r.label,
        "state":             state,
        "reason":            (d / "_failed").read_text() if state == "failed" else "",
        "missing_frames":    int((d / "_done").read_text() or 0) if state == "done" else None,
        "crops":             len(list(d.glob(f"*.{IMAGE_FORMAT}"))) if d.exists() else 0,
        "frames_no_face":    sum(n == 0 for n in nf),
        "frames_multi_face": sum(n >= 2 for n in nf),
        "frames_detected":   len(nf),
    })
status = pd.DataFrame(status_rows)
status.to_csv(RUN_DIR / "extraction_status.csv", index=False)

print("state per class:")
display(status.groupby(["label", "state"]).size().unstack(fill_value=0))

failed = status[status.state == "failed"]
if len(failed):
    print(f"failed videos ({len(failed)} total, first 20):")
    display(failed[["video_id", "label", "reason"]].head(20))

done = status[status.state == "done"]
print(f"crops per done video: min {done.crops.min()}, "
      f"median {done.crops.median():.0f}, max {done.crops.max()}")
print(f"frames without a face (box taken from nearest): "
      f"{done.frames_no_face.sum()} of {done.frames_detected.sum()}")
print(f"frames with >= 2 confident faces: "
      f"{done.frames_multi_face.sum()} / {done.frames_detected.sum()} "
      f"({100 * done.frames_multi_face.sum() / max(done.frames_detected.sum(), 1):.1f}%)")

## 6. Index CSV and summary

One row per crop. Columns: `label` (0 real, 1 fake), `video_id`, `frame_idx` (key 0–31), `src_file` (original filename in the frames folder), `n_faces`, `path` (relative to `faces_celebdf/`).

In [ ]:
# 6.1 Build the index
rows_idx = []
for r in videos[videos.video_id.isin(done.video_id)].itertuples():
    d    = out_dir_of(r.video_id)
    meta = load_boxes(r.video_id)
    if meta is None:
        continue
    for c in sorted(d.glob(f"*.{IMAGE_FORMAT}")):
        k = str(int(c.stem))
        rows_idx.append({
            "label":     r.label,
            "video_id":  r.video_id,
            "frame_idx": int(k),
            "src_file":  meta["src_file"].get(k, ""),
            "n_faces":   meta["n_faces"].get(k, 0),
            "path":      c.relative_to(FACES_ROOT).as_posix(),
        })
index_part_a = pd.DataFrame(rows_idx)
index_part_a.to_csv(INDEX_CSV, index=False)
print(f"saved {len(index_part_a):,} rows -> {INDEX_CSV}")
display(index_part_a.groupby("label").agg(
    videos=("video_id", "nunique"),
    crops=("path", "size")
))

summary_a = {
    "videos_real":          n_real,
    "videos_fake":          n_fake,
    "done_real":            int(((status.state == "done") & (status.label == 0)).sum()),
    "done_fake":            int(((status.state == "done") & (status.label == 1)).sum()),
    "failed":               failed[["video_id", "reason"]].to_dict("records"),
    "crops":                int(len(index_part_a)),
    "frames_no_face":       int(done.frames_no_face.sum()),
    "frames_multi_face_pct": round(
        100 * done.frames_multi_face.sum() / max(done.frames_detected.sum(), 1), 2
    ),
    "pipeline": PIPELINE,
}
(RUN_DIR / "summary_preprocessing.json").write_text(json.dumps(summary_a, indent=2))
print(json.dumps(
    {k: v for k, v in summary_a.items() if k not in ("pipeline", "failed")},
    indent=2
))

In [ ]:
# 6.2 Example crops: 4 real + 4 fake videos (frame key ~16)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
rng_show = random.Random(SEED + 1)
for row_axes, label in zip(axes, (0, 1)):
    vids_show = rng_show.sample(
        index_part_a[index_part_a.label == label].video_id.drop_duplicates().tolist(),
        min(4, index_part_a[index_part_a.label == label].video_id.nunique())
    )
    for ax, vid in zip(row_axes, vids_show):
        sub = index_part_a[index_part_a.video_id == vid]
        r   = sub.iloc[(sub.frame_idx - 16).abs().argmin()]
        ax.imshow(Image.open(FACES_ROOT / r.path)); ax.axis("off")
        ax.set_title(f"{'FAKE' if label else 'REAL'}\n{vid[:20]}\nfaces={r.n_faces}", fontsize=7)
plt.suptitle("Sample crops (frame ~16)", fontsize=9)
plt.tight_layout(); plt.show()

---
# Part B · Cross-dataset evaluation (no training)

## 7. Model loading and load checks

### 7.1 Settings (Part B)

**Question.** Does GBLA's FF++ advantage on the unseen generator (FaceShifter 92.91 vs 88.37 for LNCLIP-DF) carry over to Celeb-DF v2 — a dataset with a different face-swap method, different video source, and a harder real-vs-fake split?

**Models.** All 9 checkpoints from M4/M1/M6; none retrained here. All use frozen CLIP ViT-L/14 + trained LayerNorms + linear head (104,450 parameters at test time).

**Bootstrap.** 95% intervals from 2,000 resamples of **videos** (Celeb-DF v2 has no published actor/pair metadata, so we resample videos directly). Paired across models.

In [ ]:
# 7.1 Part B settings
import os
from pathlib import Path

DATA_ROOT   = Path("/nlsasfs/home/isea/isea64/GBLA-Deepfake Detection")
RUNS_ROOT   = DATA_ROOT / "runs"
FACES_CELEB = DATA_ROOT / "faces_celebdf"
CELEB_INDEX = DATA_ROOT / "index" / "celebdf_v2.csv"
FACES_FF    = DATA_ROOT / "faces"
M8B_DIR     = RUNS_ROOT / "m8b"
M9B_DIR     = RUNS_ROOT / "m9b"
M9D_DIR     = RUNS_ROOT / "m9d"

ARCH, PRETRAINED = "ViT-L-14", "openai"
L2_NORM      = True
SEED_EVAL    = 0
EPOCH        = 10
EVAL_BATCH   = 256
LOAD_THREADS = 16
N_BOOT       = 2000
N_LOAD_CHECK = 256

PAPER_CELEB_VIDEO_AUC = 89.4   # GenD (CLIP ViT-L/14), Table 4, ~mean of 5 seeds
PAPER_DFD_VIDEO_AUC   = 96.4
PAPER_UADFV_VIDEO_AUC = 99.2

M1_DIR    = RUNS_ROOT / "m1"
M4_DIR    = RUNS_ROOT / "m4"
M6_MERGES = RUNS_ROOT / "m6" / "merges"


def m4_run(tag):
    return M4_DIR / f"gbla_lam{tag}_{ARCH}_seed{SEED_EVAL}"

def m1_dir(seed=0):
    return M1_DIR / f"lnclip_{ARCH}_train-DF-F2F-FS-NT_seed{seed}"


MODELS = {
    "m1":        ("LNCLIP-DF repro (M1)",   "state",
                  m1_dir(0) / f"epoch_{EPOCH:02d}.pt",
                  m1_dir(0) / f"test_predictions_epoch{EPOCH:02d}.csv"),
    "lam_inf":   ("pooled, λ=∞ (M4)",       "gbla",
                  m4_run("inf")  / f"epoch_{EPOCH:02d}.pt",
                  m4_run("inf")  / "test_predictions.csv"),
    "lam_1":     ("GBLA λ=1",               "gbla",
                  m4_run("1")    / f"epoch_{EPOCH:02d}.pt",
                  m4_run("1")    / "test_predictions.csv"),
    "lam_0.1":   ("GBLA λ=0.1",             "gbla",
                  m4_run("0.1")  / f"epoch_{EPOCH:02d}.pt",
                  m4_run("0.1")  / "test_predictions.csv"),
    "lam_0.01":  ("GBLA λ=0.01 (ours)",     "gbla",
                  m4_run("0.01") / f"epoch_{EPOCH:02d}.pt",
                  m4_run("0.01") / "test_predictions.csv"),
    "lam_0.001": ("GBLA λ=0.001",           "gbla",
                  m4_run("0.001")/ f"epoch_{EPOCH:02d}.pt",
                  m4_run("0.001")/ "test_predictions.csv"),
    "lam_0":     ("GBLA λ=0",               "gbla",
                  m4_run("0")    / f"epoch_{EPOCH:02d}.pt",
                  m4_run("0")    / "test_predictions.csv"),
    "uniform":   ("uniform merge (M6)",      "flat",
                  M6_MERGES / "hold-FSh_uniform.pt", None),
    "fisher":    ("Fisher merge (M6)",       "flat",
                  M6_MERGES / "hold-FSh_fisher.pt",  None),
}
M6_FSH_VIDEO_AUC = {"uniform": 81.43, "fisher": 88.99}
M1_FSH_VIDEO_AUC = 88.19
MAIN = ["m1", "lam_inf", "lam_0.01"]
OURS = "lam_0.01"

assert CELEB_INDEX.exists(), (
    f"index not found — run Part A (sections 1–6) first: {CELEB_INDEX}"
)
M9D_DIR.mkdir(parents=True, exist_ok=True)
missing = [k for k, (_, _, path, _) in MODELS.items() if not path.exists()]
for k in missing:
    print(f"checkpoint not found, skipped: {k} -> {MODELS[k][2]}")
MODELS = {k: v for k, v in MODELS.items() if k not in missing}
assert all(k in MODELS for k in MAIN), "a main model is missing; check M4/M1/M6 runs"
print("models:", list(MODELS))

### 7.2 Imports and environment (Part B)

In [ ]:
# 7.2 Imports and environment (Part B)
import json, random, time
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import open_clip
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from PIL import Image
from tqdm.auto import tqdm

random.seed(SEED_EVAL); np.random.seed(SEED_EVAL); torch.manual_seed(SEED_EVAL)
DEVICE    = "cuda"
CLIP_MEAN = (0.48145466, 0.4578275,  0.40821073)
CLIP_STD  = (0.26862954, 0.26130258, 0.27577711)
display(pd.Series({
    "torch":     torch.__version__,
    "open_clip": open_clip.__version__,
    "GPU":       torch.cuda.get_device_name(0),
}, name="value").to_frame())

### 7.3 Detector and parameter report

In [ ]:
# 7.3 Detector (identical to M1 / M4 / M8b / M9b)
class Detector(nn.Module):
    def __init__(self):
        super().__init__()
        encoder     = open_clip.create_model(
            ARCH, pretrained=PRETRAINED, force_quick_gelu=True
        ).visual
        encoder.proj = None
        self.encoder = encoder
        self.head    = nn.Linear(encoder.transformer.width, 2)

    def forward(self, x):
        feats = self.encoder(x).float()
        f     = F.normalize(feats, dim=-1)
        return self.head(f if L2_NORM else feats), f


torch.manual_seed(SEED_EVAL)
model = Detector().to(DEVICE).eval()
model.requires_grad_(False)
for m in model.encoder.modules():
    if isinstance(m, nn.LayerNorm):
        m.requires_grad_(True)
model.head.requires_grad_(True)

PARAMS      = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
PARAM_NAMES = [n for n, _ in PARAMS]
P           = sum(p.numel() for _, p in PARAMS)
TOTAL       = sum(p.numel() for p in model.parameters())
param_report = {
    "total":                TOTAL,
    "trainable (deployed)": P,
    "frozen":               TOTAL - P,
    "trainable %":          round(100 * P / TOTAL, 4),
}
display(pd.Series(param_report, name="parameters").to_frame())
assert P == 104_450, f"expected 104450 trainable parameters, got {P}"


@torch.no_grad()
def set_flat(theta):
    theta  = theta.to(DEVICE).float()
    assert theta.numel() == P, (theta.numel(), P)
    offset = 0
    for _, p in PARAMS:
        n = p.numel()
        p.copy_(theta[offset:offset + n].view_as(p).to(p.dtype))
        offset += n


def load_ckpt(path):
    return torch.load(path, map_location="cpu", weights_only=False)


def load_theta(kind, path):
    obj = load_ckpt(path)
    if kind == "state":
        state = obj.get("trainable", obj)
        return torch.cat([state[n].reshape(-1).float() for n in PARAM_NAMES])
    if kind == "gbla":
        return obj.get("gbla", obj)["theta_s"].float()
    if kind == "flat":
        return obj["theta"].float()
    raise ValueError(kind)


THETAS = {k: load_theta(kind, path)
          for k, (_, kind, path, _) in MODELS.items()}
print({k: f"{v.numel():,}" for k, v in THETAS.items()})

In [ ]:
# 7.4 Prediction and metric helpers
MEAN_GPU = torch.tensor(CLIP_MEAN, device=DEVICE).view(1, 3, 1, 1)
STD_GPU  = torch.tensor(CLIP_STD,  device=DEVICE).view(1, 3, 1, 1)


def read_rgb(path):
    with Image.open(path) as im:
        return np.asarray(im.convert("RGB"))


def load_images(paths, desc):
    images = np.empty((len(paths), 224, 224, 3), dtype=np.uint8)
    def fill(i): images[i] = read_rgb(paths[i])
    with ThreadPoolExecutor(LOAD_THREADS) as pool:
        list(tqdm(pool.map(fill, range(len(paths)), chunksize=64),
                  total=len(paths), desc=desc))
    return images


@torch.no_grad()
def predict(theta, images, desc="predict"):
    set_flat(theta)
    probs = []
    for s in tqdm(range(0, len(images), EVAL_BATCH), desc=desc, leave=False):
        x = torch.from_numpy(images[s:s + EVAL_BATCH]).to(DEVICE, non_blocking=True)
        x = (x.permute(0, 3, 1, 2).float() / 255 - MEAN_GPU) / STD_GPU
        with torch.autocast("cuda", dtype=torch.bfloat16):
            logits, _ = model(x)
        probs.append(logits.float().softmax(1)[:, 1].cpu())
    return torch.cat(probs).numpy()


def auc(labels, scores):
    labels = np.asarray(labels)
    ranks  = pd.Series(np.asarray(scores)).rank().values
    n_pos  = labels.sum()
    n_neg  = len(labels) - n_pos
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    return float((ranks[labels == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


def average_precision(labels, scores):
    order = np.argsort(-np.asarray(scores), kind="stable")
    y     = np.asarray(labels)[order]
    prec  = np.cumsum(y) / np.arange(1, len(y) + 1)
    return float((prec * y).sum() / y.sum())


def roc_points(labels, scores):
    order = np.argsort(-np.asarray(scores), kind="stable")
    y     = np.asarray(labels)[order]
    tpr   = np.concatenate([[0], np.cumsum(y) / y.sum()])
    fpr   = np.concatenate([[0], np.cumsum(1 - y) / (1 - y).sum()])
    return fpr, tpr


def eer(labels, scores):
    fpr, tpr = roc_points(labels, scores)
    i = np.argmin(np.abs((1 - tpr) - fpr))
    return float((fpr[i] + 1 - tpr[i]) / 2)

### 7.5 Load check: reproduce saved FF++ test predictions

Same check as M8b/M9b. Models with saved per-crop FF++ test predictions: recompute 256 random crops and verify max |difference| < 0.05. Models without (M1 epoch-10 and M6 merges): recompute FF++ FaceShifter video AUC and compare with references (M1 88.19, uniform 81.43, Fisher 88.99; tolerance 0.3). Notebook stops if any check fails.

In [ ]:
# 7.5 Load check
ff_index = pd.read_csv(DATA_ROOT / "index" / "ffpp_c23.csv")
ff_fsh   = ff_index[
    (ff_index.split == "test") &
    ff_index.generator.isin(["original", "FaceShifter"])
].reset_index(drop=True)

FSH_REFERENCE  = {"m1": M1_FSH_VIDEO_AUC, **M6_FSH_VIDEO_AUC}
ff_fsh_images  = None
check_rows, FF_FSH_RECOMPUTED = [], {}

for k, (name, _, _, ref_path) in MODELS.items():
    if ref_path is None or not ref_path.exists():
        if ff_fsh_images is None:
            ff_fsh_images = load_images(
                [str(FACES_FF / p) for p in ff_fsh.path], desc="FF++ test real+FSh"
            )
        pred = ff_fsh.assign(
            prob=predict(THETAS[k], ff_fsh_images, desc=f"FF++ FSh {k}")
        )
        vid  = pred.groupby(["generator", "video_id"], as_index=False).agg(
            label=("label", "first"), prob=("prob", "mean")
        )
        FF_FSH_RECOMPUTED[k] = 100 * auc(vid.label, vid.prob)
        expected = FSH_REFERENCE.get(k)
        check_rows.append({
            "model":               k,
            "crops":               len(ff_fsh),
            "FF++ FSh video AUC":  round(FF_FSH_RECOMPUTED[k], 2),
            "expected":            expected,
            "ok": bool(expected is None or
                       abs(FF_FSH_RECOMPUTED[k] - expected) < 0.3),
        })
        continue
    ref  = pd.read_csv(ref_path).sample(N_LOAD_CHECK, random_state=SEED_EVAL)
    imgs = load_images([str(FACES_FF / p) for p in ref.path], desc=f"check {k}")
    diff = np.abs(predict(THETAS[k], imgs, desc=f"check {k}") - ref.prob.values)
    check_rows.append({
        "model":        k,
        "crops":        len(ref),
        "max |diff|":   round(float(diff.max()),  4),
        "mean |diff|":  round(float(diff.mean()), 5),
        "ok": bool(diff.max() < 0.05),
    })

load_check = pd.DataFrame(check_rows)
display(load_check)
load_check.to_csv(M9D_DIR / "load_check.csv", index=False)
assert load_check.ok.all(), "a checkpoint does not reproduce its FF++ results — stop"
del ff_fsh_images

## 8. Load Celeb-DF crops into RAM

Celeb-DF v2 has ~209,856 crops (6,533 done videos × 32 frames). At 224×224×3 bytes that is roughly **31 GB**. Confirm available RAM before loading. If memory is tight, reduce `EVAL_BATCH` or evaluate one model at a time and cache predictions to disk.

In [ ]:
# 8. Load all Celeb-DF crops into RAM
index_celeb  = pd.read_csv(CELEB_INDEX, dtype={"video_id": str})
videos_celeb = index_celeb.groupby("video_id", as_index=False).agg(
    label=("label", "first"),
    crops=("path",  "size"),
)
print(f"crops : {len(index_celeb):,}   "
      f"videos: real {(videos_celeb.label == 0).sum()}, "
      f"fake {(videos_celeb.label == 1).sum()}")
print(f"estimated RAM: {len(index_celeb) * 224 * 224 * 3 / 1e9:.1f} GB")

t0 = time.time()
CELEB_IMAGES = load_images(
    [str(FACES_CELEB / p) for p in index_celeb.path],
    desc="Celeb-DF crops"
)
print(f"loaded {CELEB_IMAGES.nbytes / 1e9:.1f} GB in {(time.time() - t0) / 60:.1f} min")

## 9. Predict every model (resumable)

Frame probabilities go to `runs/m9d/pred_<model>.csv` (same row order as the index). Existing files are reused.

In [ ]:
# 9. Predict
FRAME_PROBS = {}
for k, (name, *_) in MODELS.items():
    path = M9D_DIR / f"pred_{k}.csv"
    if path.exists():
        saved = pd.read_csv(path)
        if (len(saved) == len(index_celeb) and
                (saved.path.values == index_celeb.path.values).all()):
            FRAME_PROBS[k] = saved.prob.values
            print(f"{k}: reused {path.name}")
            continue
    t0    = time.time()
    probs = predict(THETAS[k], CELEB_IMAGES, desc=k)
    pd.DataFrame({
        "video_id":  index_celeb.video_id,
        "frame_idx": index_celeb.frame_idx,
        "path":      index_celeb.path,
        "prob":      probs,
    }).to_csv(path, index=False)
    FRAME_PROBS[k] = probs
    print(f"{k}: predicted in {time.time() - t0:.0f} s")

## 10. Results on Celeb-DF v2

### 10.1 Main table

Video score = mean frame probability (paper protocol). `real acc` / `fake acc` use threshold 0.5. CI = 95% video-level bootstrap (2,000 resamples; Celeb-DF v2 has no published actor/pair metadata for stratified bootstrap).

In [ ]:
# 10.1 Main table
def video_scores(k, frame_mask=None):
    df   = pd.DataFrame({"video_id": index_celeb.video_id,
                          "prob":     FRAME_PROBS[k]})
    allv = df.groupby("video_id").prob.mean()
    if frame_mask is not None:
        allv = (df[frame_mask]
                .groupby("video_id").prob.mean()
                .reindex(allv.index)
                .fillna(allv))
    return allv.reindex(videos_celeb.video_id).values


VIDEO_SCORES = {k: video_scores(k) for k in MODELS}
y_vid        = videos_celeb.label.values

# bootstrap: resample videos
all_vids = np.arange(len(videos_celeb))
rng_boot = np.random.default_rng(SEED_EVAL)
BOOT_ROWS = [
    rng_boot.choice(all_vids, len(all_vids), replace=True)
    for _ in range(N_BOOT)
]


def boot_auc(scores):
    return np.array([auc(y_vid[rows], scores[rows]) for rows in BOOT_ROWS])


BOOT = {k: boot_auc(VIDEO_SCORES[k]) for k in tqdm(MODELS, desc="bootstrap")}

rows_tbl = []
for k, (name, *_) in MODELS.items():
    s      = VIDEO_SCORES[k]
    lo, hi = np.nanpercentile(BOOT[k], [2.5, 97.5]) * 100
    rows_tbl.append({
        "model":     name,
        "key":       k,
        "video AUC": 100 * auc(y_vid, s),
        "95% CI":    f"[{lo:.2f}, {hi:.2f}]",
        "frame AUC": 100 * auc(index_celeb.label.values, FRAME_PROBS[k]),
        "video AP":  100 * average_precision(y_vid, s),
        "video EER": 100 * eer(y_vid, s),
        "real acc":  100 * float((s[y_vid == 0] <= 0.5).mean()),
        "fake acc":  100 * float((s[y_vid == 1] >  0.5).mean()),
    })
rows_tbl.append({
    "model": "GenD (CLIP) paper, 5 seeds", "key": "paper",
    "video AUC": PAPER_CELEB_VIDEO_AUC,
})
main_table = pd.DataFrame(rows_tbl).set_index("model")
display(main_table.round(2))
main_table.to_csv(M9D_DIR / "celebdf_main_table.csv")
print(f"\nall models: {P:,} trainable parameters at test time "
      f"({param_report['trainable %']}% of {TOTAL:,})")

### 10.2 Paired differences (ours − other)

Same bootstrap resamples for both models, so the interval covers the difference itself. `P(diff > 0)` = fraction of resamples where GBLA λ=0.01 wins.

In [ ]:
# 10.2 Paired differences
diff_rows = []
for k in MODELS:
    if k == OURS:
        continue
    d       = (BOOT[OURS] - BOOT[k]) * 100
    lo, hi  = np.nanpercentile(d, [2.5, 97.5])
    diff_rows.append({
        "ours − ":        MODELS[k][0],
        "video AUC diff": 100 * (auc(y_vid, VIDEO_SCORES[OURS])
                                  - auc(y_vid, VIDEO_SCORES[k])),
        "95% CI":         f"[{lo:+.2f}, {hi:+.2f}]",
        "P(diff > 0)":    float(np.nanmean(d > 0)),
    })
diffs = pd.DataFrame(diff_rows).set_index("ours − ")
display(diffs.round(3))
diffs.to_csv(M9D_DIR / "celebdf_paired_differences.csv")

## 11. Cross-dataset λ curve

Shows all four OOD evaluations on a single axis: FF++ FaceShifter (unseen generator within FF++), DFD, UADFV, and Celeb-DF v2 — plotted against λ. FF++ numbers come from each model's saved test predictions; DFD and UADFV are read from M8b/M9b summary CSVs (read-only).

In [ ]:
# 11. Cross-dataset λ curve
SEEN = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]


def ff_video_auc(pred, gens):
    sub = pred[pred.generator.isin(["original", *gens])]
    vid = sub.groupby(["generator", "video_id"], as_index=False).agg(
        label=("label", "first"), prob=("prob", "mean")
    )
    return 100 * auc(vid.label, vid.prob)


# read DFD / UADFV tables if available
dfd_table   = None
uadfv_table = None
if (M8B_DIR / "dfd_main_table.csv").exists():
    dfd_table = pd.read_csv(M8B_DIR / "dfd_main_table.csv").set_index("key")
if (M9B_DIR / "uadfv_main_table.csv").exists():
    uadfv_table = pd.read_csv(M9B_DIR / "uadfv_main_table.csv").set_index("key")

cmp_rows = []
for k, (name, _, _, ref_path) in MODELS.items():
    row = {"model": name}
    if ref_path is not None and ref_path.exists():
        pred = pd.read_csv(ref_path)
        pred = pred[pred.split == "test"] if "split" in pred else pred
        row["FF++ seen (DF+F2F+FS+NT)"] = ff_video_auc(pred, SEEN)
        row["FF++ NT"]                  = ff_video_auc(pred, ["NeuralTextures"])
        row["FF++ FaceShifter (unseen)"]= ff_video_auc(pred, ["FaceShifter"])
    elif k in FF_FSH_RECOMPUTED:
        row["FF++ FaceShifter (unseen)"]= FF_FSH_RECOMPUTED[k]
    if dfd_table is not None and k in dfd_table.index:
        row["DFD video AUC"]   = float(dfd_table.loc[k, "video AUC"])
    if uadfv_table is not None and k in uadfv_table.index:
        row["UADFV video AUC"] = float(uadfv_table.loc[k, "video AUC"])
    row["Celeb-DF v2 video AUC"] = 100 * auc(y_vid, VIDEO_SCORES[k])
    cmp_rows.append(row)

# add paper reference row
cmp_rows.append({
    "model":                      "GenD (CLIP) paper, 5 seeds",
    "FF++ FaceShifter (unseen)":  86.6,
    "DFD video AUC":              PAPER_DFD_VIDEO_AUC,
    "UADFV video AUC":            PAPER_UADFV_VIDEO_AUC,
    "Celeb-DF v2 video AUC":      PAPER_CELEB_VIDEO_AUC,
})

cross = pd.DataFrame(cmp_rows).set_index("model")
display(cross.round(2))
cross.to_csv(M9D_DIR / "cross_dataset_summary.csv")

# λ curve plot
lam_keys = [k for k in
             ["lam_inf", "lam_1", "lam_0.1", "lam_0.01", "lam_0.001", "lam_0"]
             if k in MODELS]
labels = [k.replace("lam_", "") for k in lam_keys]

fig, ax = plt.subplots(figsize=(8, 4))
plot_cols = [
    ("Celeb-DF v2 video AUC",     "o", "C0"),
    ("DFD video AUC",             "D", "C1"),
    ("UADFV video AUC",           "s", "C2"),
    ("FF++ FaceShifter (unseen)", "^", "C3"),
]
for col, marker, color in plot_cols:
    if col in cross.columns:
        vals = []
        for k in lam_keys:
            name = MODELS[k][0]
            vals.append(cross.loc[name, col] if name in cross.index else float("nan"))
        ax.plot(labels, vals, marker=marker, color=color, label=col, linewidth=1.4)

ax.axhline(PAPER_CELEB_VIDEO_AUC, ls="--", c="C0", lw=1,
           label=f"paper Celeb-DF v2 ({PAPER_CELEB_VIDEO_AUC})")
ax.set_xlabel("λ  (inf = pooled ERM)")
ax.set_ylabel("video AUC %")
ax.legend(fontsize=8, loc="best")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(M9D_DIR / "lambda_curve.png", dpi=150)
plt.show()

if "DFD video AUC" in cross.columns:
    sub = cross.iloc[:-1].dropna(subset=["DFD video AUC"])
    rho_dfd = sub["DFD video AUC"].corr(
        sub["Celeb-DF v2 video AUC"], method="spearman"
    )
    print(f"rank agreement DFD vs Celeb-DF v2 (Spearman ρ): {rho_dfd:.2f} "
          f"over {len(sub)} models")
if "UADFV video AUC" in cross.columns:
    sub = cross.iloc[:-1].dropna(subset=["UADFV video AUC"])
    rho_ua = sub["UADFV video AUC"].corr(
        sub["Celeb-DF v2 video AUC"], method="spearman"
    )
    print(f"rank agreement UADFV vs Celeb-DF v2 (Spearman ρ): {rho_ua:.2f} "
          f"over {len(sub)} models")

## 12. ROC curves and summary

In [ ]:
# 12.1 ROC curves (video level)
fig, ax = plt.subplots(figsize=(5, 4.8))
for k in MAIN + [k for k in ("fisher", "uniform") if k in MODELS]:
    fpr, tpr = roc_points(y_vid, VIDEO_SCORES[k])
    ax.plot(fpr, tpr,
            label=f"{MODELS[k][0]} ({100 * auc(y_vid, VIDEO_SCORES[k]):.2f})",
            lw=1.4)
ax.plot([0, 1], [0, 1], "k:", lw=0.8)
ax.set_xlabel("false positive rate (real called fake)")
ax.set_ylabel("true positive rate")
ax.set_title("Celeb-DF v2, video level")
ax.legend(fontsize=7, loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(M9D_DIR / "celebdf_roc.png", dpi=150)
plt.show()

In [ ]:
# 12.2 Summary JSON
summary_b = {
    "protocol":    ("train FF++ c23 (DF, F2F, FS, NT), "
                    "test Celeb-DF v2 888 real + 5645 fake, "
                    "video AUC = mean frame prob"),
    "parameters":  param_report,
    "paper_gend_clip_celebdf_video_auc": PAPER_CELEB_VIDEO_AUC,
    "load_check":  load_check.astype(str).to_dict("records"),
    "celebdf":     (main_table
                    .drop(columns=["key"], errors="ignore")
                    .round(3).astype(str).to_dict("index")),
    "paired_differences_ours_minus": diffs.round(3).astype(str).to_dict("index"),
    "cross_dataset": cross.round(3).to_dict("index"),
    "bootstrap":   (f"{N_BOOT} video-level resamples "
                    f"({len(videos_celeb)} videos), seed {SEED_EVAL}"),
    "seeds_per_model": 1,
}
(M9D_DIR / "summary.json").write_text(
    json.dumps(summary_b, indent=2, ensure_ascii=False)
)
print(json.dumps(
    {k: summary_b[k] for k in ("celebdf", "paired_differences_ours_minus")},
    indent=1, ensure_ascii=False
))